# SciFact results in MLflow

`python -m ragbench.evaluation.runner --config configs/scifact.yaml` writes one MLflow run per pipeline under the experiment **`ragbench/scifact`**. The same numbers are also saved to `data/scifact/eval_latest.json` for the API.

MLflow metric names use `_at_` in place of `@` (`ndcg_at_10` is NDCG@10). Latency fields are milliseconds for a single query.

The tracking server is `http://localhost:5001` unless `MLFLOW_TRACKING_URI` is set (`docker compose up -d mlflow`).

In [1]:
import os
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

import mlflow
import pandas as pd
from IPython.display import display

tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5001")
mlflow.set_tracking_uri(tracking_uri)

experiment = mlflow.get_experiment_by_name("ragbench/scifact")
if experiment is None:
    raise SystemExit("No experiment named ragbench/scifact. Run the SciFact eval with MLflow up.")

runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
print(f"{experiment.name}  ({len(runs)} runs)  tracking={tracking_uri}")

# Re-running eval appends runs. Keep the latest finished run of each pipeline.
finished = runs[runs["status"] == "FINISHED"].sort_values("start_time")
latest = finished.groupby("tags.pipeline", as_index=False).tail(1)

HEADLINE = [
    "ndcg_at_10",
    "mrr_at_10",
    "recall_at_10",
    "recall_at_100",
    "map_at_100",
    "precision_at_10",
    "p50_ms",
    "p95_ms",
    "p99_ms",
    "mean_ms",
]
table = latest[["tags.pipeline", "tags.embedding_model", "start_time"] + [f"metrics.{name}" for name in HEADLINE]].copy()
table.columns = ["pipeline", "embedding_model", "start_time"] + HEADLINE
table = table.sort_values("ndcg_at_10", ascending=False).reset_index(drop=True)
table["start_time"] = pd.to_datetime(table["start_time"], utc=True).dt.strftime("%Y-%m-%d %H:%M")
display(table.round(4))

ragbench/scifact  (4 runs)  tracking=http://localhost:5001


,pipeline,embedding_model,start_time,ndcg_at_10,mrr_at_10,recall_at_10,recall_at_100,map_at_100,precision_at_10,p50_ms,p95_ms,p99_ms,mean_ms
0,dense,BAAI/bge-large-en-v1.5,2026-09-25 19:40,0.7410,0.7047,0.8766,0.9517,0.6982,0.0997,170.985,282.766,382.484,185.455
1,hybrid,BAAI/bge-large-en-v1.5,2026-09-25 19:41,0.7205,0.6888,0.8452,0.9583,0.6810,0.0957,201.888,288.141,360.827,210.709
2,bm25,BAAI/bge-large-en-v1.5,2026-09-25 19:38,0.6863,0.6492,0.8187,0.9127,0.6439,0.0907,0.179,0.470,0.613,0.249
3,reranked,BAAI/bge-large-en-v1.5,2026-09-25 22:24,0.6662,0.6318,0.7956,0.9370,0.6259,0.0893,28645.857,38048.277,43232.387,29390.271


**How to read the table**

- **NDCG@10**, **MRR@10**, **MAP@100** — ranking quality. Higher means relevant abstracts show up earlier.
- **Recall@10 / Recall@100** — fraction of the gold abstracts that appear in the top 10 or top 100.
- **Precision@10** — fraction of the top 10 that are gold. On SciFact this stays near 0.1 on every pipeline, because most claims have a single relevant abstract, so nine of the ten hits are non-relevant by construction.
- **p50_ms / p95_ms** — median and tail latency of one retrieval call.

Dense, hybrid, and reranked all embed with `BAAI/bge-large-en-v1.5`. BM25 does not use that model; the tag is still logged because it is part of the dataset config.

In [2]:
best = table.iloc[0]
fastest = table.sort_values("p50_ms").iloc[0]
best_recall = table.sort_values("recall_at_100", ascending=False).iloc[0]
print(
    f"Highest NDCG@10: {best.pipeline} ({best.ndcg_at_10:.3f}).\n"
    f"Highest Recall@100: {best_recall.pipeline} ({best_recall.recall_at_100:.3f}).\n"
    f"Fastest p50: {fastest.pipeline} ({fastest.p50_ms:.1f} ms)."
)

extra = [
    col.split(".", 1)[1]
    for col in latest.columns
    if col.startswith("metrics.") and col.split(".", 1)[1] not in HEADLINE
]
if extra:
    print("Additional metrics on these runs:", ", ".join(sorted(extra)))
else:
    print("No RAGAS metrics on these runs (the eval was logged with retrieval metrics only).")

Highest NDCG@10: dense (0.741).
Highest Recall@100: hybrid (0.958).
Fastest p50: bm25 (0.2 ms).
No RAGAS metrics on these runs (the eval was logged with retrieval metrics only).
